In [3]:
!pip -q install chromadb sentence-transformers langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95

Our knowledge base

In [4]:
documents = [
    """
    Python generators produce values lazily. Instead of creating all values
    in memory at once, a generator produces values one at a time when they
    are requested. This makes generators useful for processing large datasets
    while reducing memory usage.
    """,

    """
    Python generator functions use the yield keyword. Calling a generator
    function returns a generator object. Execution pauses at each yield and
    resumes when another value is requested.
    """,

    """
    FastAPI is a Python web framework designed for building APIs. It uses
    Python type hints for request validation and can automatically generate
    OpenAPI documentation.
    """,

    """
    Redis is an in-memory data store commonly used for caching, queues,
    sessions, and other applications that require fast data access.
    """,

    """
    Docker packages an application together with its dependencies into
    containers. Containers help applications run consistently across
    different environments.
    """
]

Embed the documents

In [6]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

document_embedding = embedding_model.encode(documents)

print(document_embedding.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(5, 384)


Create Chromadb

In [8]:
import chromadb

Client = chromadb.Client()

collection = Client.get_or_create_collection(
    name = "day_07_collection"
)

In [9]:
collection.add(
    ids=[f"doc_{i}" for i in range(len(documents))],
    documents=documents,
    embeddings=document_embedding.tolist()
)

Build the RETRIEVAL part

In [15]:
query = "How do Python generators save memory?"

In [17]:
query_embedding = embedding_model.encode([query])[0]

In [21]:
results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=2,
    include=["documents", "distances"]
)

In [22]:
for i, (doc, distance) in enumerate(
    zip(
        results["documents"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\nResult {i}")
    print(f"Distance: {distance:.4f}")
    print(doc)


Result 1
Distance: 0.4021

    Python generators produce values lazily. Instead of creating all values
    in memory at once, a generator produces values one at a time when they
    are requested. This makes generators useful for processing large datasets
    while reducing memory usage.
    

Result 2
Distance: 0.6519

    Python generator functions use the yield keyword. Calling a generator
    function returns a generator object. Execution pauses at each yield and
    resumes when another value is requested.
    


Build the context

In [26]:
retrived_documents = results["documents"][0]

context = "\n\n".join(retrived_documents)

print(context)


    Python generators produce values lazily. Instead of creating all values
    in memory at once, a generator produces values one at a time when they
    are requested. This makes generators useful for processing large datasets
    while reducing memory usage.
    


    Python generator functions use the yield keyword. Calling a generator
    function returns a generator object. Execution pauses at each yield and
    resumes when another value is requested.
    


Build the prompt

In [41]:
prompt = f"""
You are a helpful AI assistant.

Answer the user's question using ONLY the provided context.

If the answer cannot be found in the context, say:
"I don't have enough information in the provided documents."

Do not use outside knowledge.

Context:
{context}

Question:
{query}

Answer:
"""

Send it to Gemini

In [28]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

In [34]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=GEMINI_API_KEY
)

In [35]:
response = llm.invoke(prompt)

print(response.content)

[{'type': 'text', 'text': 'Based on the provided context, Python generators save memory by producing values lazily—one at a time when requested—instead of creating all values in memory at once.', 'extras': {'signature': 'EtkMCtYMARFNMg87oTSMMjVsG2fQ6SeQ/27y+B9DoKTUqkYEirr71QMFTn6WmZnI9SPWV5+w2R9nBd8dRTFvdV+uwT++622vaEp+x4ngOV982F8oUoTcq67v1V/hO0nodH1yrDsHBBpFf+shP18eu26HKtBIpXvmPmDfP/pxAEAQLfsMmCHFAi4ZR9JWf+JJGhpz7pCrSzOC5f8dwzIj3AEx9sZy8IIxQ2hNUgqPqa3emCdQAtkFQqoIvbdbUAP8TkBDmckhpi4cBCukCiT6X12sI5NyFxc0KKvsZbsIq7QvHX9JFiKIvDX03GzF5THxFek7MHrBX2gkxB3dd07R88Y84tmEXDzpvxiEWcYJdj/+kaeLtL1ht4/ay5Se2BqkZ/qimeU0J41WWRk1HZL4g0aFKe04ZZFJ9LMGKMkyHjCP7j3MMYfc3CFblfUzd/6+BksRymbNfHt00h/xZp6Y2tP9FvQhDNircRUFggQUroZXsa/Y7GDUwhTb0JIyVEjKgjC7gxRG8Px284ZUNSddEGjE915NjKbGv2yagnhIJKZESxVaMtr/6D8PY7wg/6x57nTnt/XFIGHLUkO6noRw9zhq57LfIpeBI5nD40FAgtE37Q88wyo78eGv0XeOUuZdyJUtf86H0fmCt+ZWNmueV7Rdw2PPRUHJy6VIhQ/h2lr2ug44kRTcy4yjL3krBjK29baY2FLHtEqoQJLppGonUbsDJ6Wm+flu8Fw0i/vrpUPZVSlO4BbXQjnxo+U0SubZkdljaBps7Rb

In [36]:
query = "Who invented Docker?"

In [37]:
query_embedding = embedding_model.encode([query])[0]

In [38]:
results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=2,
    include=["documents", "distances"]
)

In [39]:
for i, (doc, distance) in enumerate(
    zip(
        results["documents"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"\nResult {i}")
    print(f"Distance: {distance:.4f}")
    print(doc)


Result 1
Distance: 1.1818

    Docker packages an application together with its dependencies into
    containers. Containers help applications run consistently across
    different environments.
    

Result 2
Distance: 1.7800

    Python generators produce values lazily. Instead of creating all values
    in memory at once, a generator produces values one at a time when they
    are requested. This makes generators useful for processing large datasets
    while reducing memory usage.
    


In [40]:
retrived_documents = results["documents"][0]

context = "\n\n".join(retrived_documents)

print(context)


    Docker packages an application together with its dependencies into
    containers. Containers help applications run consistently across
    different environments.
    


    Python generators produce values lazily. Instead of creating all values
    in memory at once, a generator produces values one at a time when they
    are requested. This makes generators useful for processing large datasets
    while reducing memory usage.
    


In [42]:
prompt = f"""
You are a helpful AI assistant.

Answer the user's question using ONLY the provided context.

If the answer cannot be found in the context, say:
"I don't have enough information in the provided documents."

Do not use outside knowledge.

Context:
{context}

Question:
{query}

Answer:
"""

In [43]:
response = llm.invoke(prompt)

print(response.content)

[{'type': 'text', 'text': "I don't have enough information in the provided documents.", 'extras': {'signature': 'EpEMCo4MARFNMg+bmJYLuSkhrbL2+eG67ScgSlimjWTualN7/LG7fCPnNGGS5lyQpo3ZX106A/dGn8QhOHRKjEHnsWxyhieuxoWPdQzYE49hdqNtE/X37ALPd+d8aGhVkw3UiOH+hF+B+zEoDl2MlZTPvD1pT8AerRIsATbTJYb0LYWPKEx25VvBPPP/pM0DUjko4hmOz3GtZx0WeK7+NuqB/xX48PjbIsdXuLSqgGLVYk2Q0F6jIrmkBHp41aeGSJbnTbACimHM0aD9nEFnVCDgDn3o1RHN/9o+Lt0z1t4X0sCCtKrOOU5/HS+SPorMdbfI4eydkXSEByvMjz9ViM97xvjR4WC/uVuiJ7YjAtzL80aAcq5OYcVVoG+jQu+zlEDy7S0krY0kseKTi3P57jJwhLU7dq7KN64xl38vIohDQ0MIVpcAsL+I5/33hgY+v+uf4oRBzhjWHkD8fnPlmdNKSIptAMMXEEDJJ1UNdsKKfklu8/2BS0vBXkqw3K4hXhiYCJgNvaBrEMB0paIZfyUd8Nw+7mRWExc+F+sdz4Dob3Va82aNkNKC6BfUdX4DVNAtQXloDLhoieKtxmFPfnZcVvoWLjQmfN69fyjWPKlChuEGXSw3LnbamdX/db5Tis1RMBwb54DyUfCc8ZlDAblTKoEka+PdKCWkEd/2s5BcQrk7hvnm5yRbkHJe6CQxXGJXEsX+VWLyyT+YA6tmrqhymup/RxQCyaLnKMEOOrrYXlaM3jhThs9mN1RhdVEt7fjedoVDelSMJy4NTKT/nLJ2IcYWTn12ZpHoc9g6toPikLzotqrayuKkzMcud4jJr0cVmKx0hqAI69DKbT1yixOxrZFQTT0efb9iv4vQF8FpAwr9ml2ZEQM

## Day 7 Reflection

What I learned:

- RAG stands for Retrieval-Augmented Generation.
- RAG combines information retrieval with LLM generation.
- A RAG system generally has an ingestion phase and a query phase.
- During ingestion, documents are chunked, embedded, and stored in a vector database.
- During querying, the user's question is embedded and used to retrieve relevant chunks.
- Retrieved chunks are added to the LLM prompt as context.
- The LLM generates the final response using the retrieved context.
- A vector database performs retrieval; it does not generate the final answer.
- RAG can provide external or private information to an LLM at query time.
- RAG can reduce hallucinations when retrieval and grounding work correctly,
  but it does not guarantee factual correctness.
- RAG systems should be evaluated for both retrieval quality and answer quality.

Key takeaway:

RAG is not a single library or model. It is an architecture that connects
document processing, embeddings, retrieval, context construction, and LLM
generation into one system.